In [1]:
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel

In [2]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "categorize_products_no_cate"
HF_USER = "leearum95" # your HF name here!

LITE_MODE = False

DATA_USER = "leearum95"
DATASET_NAME = f"{DATA_USER}/items_prompts_full_no_category"
if LITE_MODE:
  # RUN_NAME = "2026-05-16_15.51.46-lite"
  REVISION = None
else:
  RUN_NAME = "2026-05-08_18.44.24"
  REVISION = None



PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ_API_KEY is set.")
else:
    print("GROQ_API_KEY is not set.")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")

hf_token = os.environ['HF_TOKEN']
if hf_token:
    print("HuggingFace token found.")
else:
    print("No HuggingFace token found.")

login(hf_token, add_to_git_credential=True)

#------------------------------

openrouter_url = "https://openrouter.ai/api/v1"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GROQ_API_KEY is set.
OPENROUTER_API_KEY is set.
HuggingFace token found.


In [4]:
QUANT_4_BIT = True

if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
    device = "cuda"

elif torch.backends.mps.is_available():
    use_bf16 = False
    device = "mps"

else:
    use_bf16 = False
    device = "cpu"

print("device:", device)
print("use_bf16:", use_bf16)

device: mps
use_bf16: False


In [5]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/389M [00:00<?, ?B/s]

Memory footprint: 2975.7 MB


In [9]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

In [13]:
def llama_finetuned(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs,min_new_tokens = 2, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
def gpt_5_1(item):
    # This is a placeholder for the GPT-5 API call
    # You would replace this with the actual API call to GPT-5
    return "category_from_gpt_5"

#RAG

In [ ]:
def rag(item):
    # This is a placeholder for the RAG API call
    # You would replace this with the actual API call to your RAG system
    return "category_from_rag"